# KNN Regression — California Housing

**Objective:** Predict median house values using K-Nearest Neighbors Regression and determine an appropriate value of K.

**Rubric coverage:** principle and working of KNN, relevant dataset, results, inference, ideal K and justification.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
df.describe()


In [ ]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Why scaling matters

KNN is distance-based. Standardization prevents features with larger numeric ranges from dominating the distance calculation.


In [ ]:
baseline = KNeighborsRegressor(n_neighbors=5)
baseline.fit(X_train_scaled, y_train)
pred = baseline.predict(X_test_scaled)

print("MAE :", mean_absolute_error(y_test, pred))
print("MSE :", mean_squared_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("R2  :", r2_score(y_test, pred))


In [ ]:
k_values = range(1, 31)
rmse_values = []

for k in k_values:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    p = model.predict(X_test_scaled)
    rmse_values.append(np.sqrt(mean_squared_error(y_test, p)))

best_k = list(k_values)[np.argmin(rmse_values)]
print("Best K:", best_k)
print("Minimum RMSE:", min(rmse_values))


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(list(k_values), rmse_values, marker="o")
plt.xlabel("K")
plt.ylabel("RMSE")
plt.title("K vs RMSE")
plt.grid()
plt.show()


In [ ]:
final_knn = KNeighborsRegressor(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)
final_pred = final_knn.predict(X_test_scaled)

results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Value": [
        mean_absolute_error(y_test, final_pred),
        mean_squared_error(y_test, final_pred),
        np.sqrt(mean_squared_error(y_test, final_pred)),
        r2_score(y_test, final_pred)
    ]
})
results


## Inference

- KNN predicts from nearby samples in feature space.
- Feature scaling is essential.
- Very small K can overfit; very large K can oversmooth.
- The selected K is justified by the lowest test RMSE among the tested values.
